In [ ]:
!pip install pyspark

In [ ]:
# Part A: Setup
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# Initialize Spark - This is the most important line
spark = SparkSession.builder.appName("Week1_Project").getOrCreate()

# Part B: Define the Contract (Schema)
telecom_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("age", IntegerType(), True),
    StructField("tenure_months", IntegerType(), True),
    StructField("monthly_charges", DoubleType(), True),
    StructField("total_usage_gb", DoubleType(), True),
    StructField("churn_status", IntegerType(), True)
])

# Part C: Create the Data
data = [
    (101, 34, 12, 850.5, 150.0, 0),
    (102, 45, 2, 1200.0, 50.0, 1),
    (103, 28, 36, 450.0, 400.0, 0)
]

# Part D: Build the DataFrame
df = spark.createDataFrame(data, schema=telecom_schema)

# Part E: Show Results
print("--- Week 1: Data Successfully Validated ---")
df.show()
print(f"Validated Row Count: {df.count()}")

--- Week 1: Data Successfully Validated ---
+-----------+---+-------------+---------------+--------------+------------+
|customer_id|age|tenure_months|monthly_charges|total_usage_gb|churn_status|
+-----------+---+-------------+---------------+--------------+------------+
|        101| 34|           12|          850.5|         150.0|           0|
|        102| 45|            2|         1200.0|          50.0|           1|
|        103| 28|           36|          450.0|         400.0|           0|
+-----------+---+-------------+---------------+--------------+------------+

Validated Row Count: 3


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

# 1. Create a SQL View
# This allows us to use Spark SQL for feature engineering
df.createOrReplaceTempView("telecom_data")

# 2. Spark SQL Aggregation (Requirement)
# We calculate 'usage_intensity' = total_usage / tenure
df_featured = spark.sql("""
    SELECT *,
           (total_usage_gb / tenure_months) as usage_intensity,
           (monthly_charges / 100) as scaled_charges
    FROM telecom_data
""")

# 3. Handle Nulls (Production Requirement)
df_featured = df_featured.na.fill(0)

# 4. VectorAssembler (Requirement)
# Bundling features into one single column called 'features'
assembler = VectorAssembler(
    inputCols=["age", "tenure_months", "monthly_charges", "total_usage_gb", "usage_intensity"],
    outputCol="features"
)

# Transform the data
output = assembler.transform(df_featured)

print("--- Week 2: Feature Engineering Complete ---")
output.select("customer_id", "features", "churn_status").show(truncate=False)

--- Week 2: Feature Engineering Complete ---
+-----------+-----------------------------------------+------------+
|customer_id|features                                 |churn_status|
+-----------+-----------------------------------------+------------+
|101        |[34.0,12.0,850.5,150.0,12.5]             |0           |
|102        |[45.0,2.0,1200.0,50.0,25.0]              |1           |
|103        |[28.0,36.0,450.0,400.0,11.11111111111111]|0           |
+-----------+-----------------------------------------+------------+



In [7]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

# --- 1. Expanded Data (To ensure both Churn 0 and 1 are present) ---
data_expanded = [
    (101, 34, 12, 850.5, 150.0, 0),
    (102, 45, 2, 1200.0, 50.0, 1),
    (103, 28, 36, 450.0, 400.0, 0),
    (104, 50, 40, 300.0, 500.0, 0),
    (105, 22, 1, 1500.0, 10.0, 1),
    (106, 35, 24, 900.0, 200.0, 1)
]
df_raw = spark.createDataFrame(data_expanded, schema=telecom_schema)
df_raw.createOrReplaceTempView("telecom_data_new")

# --- 2. Feature Engineering (Spark SQL) ---
df_featured = spark.sql("""
    SELECT *,
           (total_usage_gb / tenure_months) as usage_intensity
    FROM telecom_data_new
""")

# --- 3. Vector Assembler ---
assembler = VectorAssembler(
    inputCols=["age", "tenure_months", "monthly_charges", "total_usage_gb", "usage_intensity"],
    outputCol="features"
)
output = assembler.transform(df_featured)

# --- 4. Train/Test Split (70/30) ---
train_data, test_data = output.randomSplit([0.7, 0.3], seed=42)

# --- 5. Random Forest Model ---
rf = RandomForestClassifier(featuresCol="features", labelCol="churn_status")
model = rf.fit(train_data)

# --- 6. Predictions & Evaluation ---
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="churn_status", rawPredictionCol="rawPrediction")
accuracy = evaluator.evaluate(predictions)

print("--- Week 3: Modeling Successfully Completed ---")
predictions.select("customer_id", "prediction", "churn_status").show()
print(f"Model Accuracy (AUC): {accuracy:.2%}")

--- Week 3: Modeling Successfully Completed ---
+-----------+----------+------------+
|customer_id|prediction|churn_status|
+-----------+----------+------------+
|        103|       0.0|           0|
|        104|       0.0|           0|
+-----------+----------+------------+

Model Accuracy (AUC): 0.00%


In [8]:
# 1. Save the Model (Deployment Requirement)
# This allows us to load the 'brain' later for instant predictions
model_path = "spark_churn_model"
model.write().overwrite().save(model_path)

# 2. Save the Results as Parquet (Storage Requirement)
# Parquet is a columnar storage format used in 2TB+ environments
predictions.select("customer_id", "features", "prediction") \
    .write.mode("overwrite").parquet("final_predictions.parquet")

print("--- Week 4: Deployment & Storage Complete ---")
print(f"✅ Model saved to: {model_path}")
print(f"✅ Data saved as: final_predictions.parquet")

# 3. Final Verification: List the files
import os
print("\nFiles in your Distributed Environment:")
print(os.listdir())

--- Week 4: Deployment & Storage Complete ---
✅ Model saved to: spark_churn_model
✅ Data saved as: final_predictions.parquet

Files in your Distributed Environment:
['.config', 'spark_churn_model', 'final_predictions.parquet', 'sample_data']


In [9]:
import shutil

# Zip the model folder
shutil.make_archive('spark_churn_model_final', 'zip', 'spark_churn_model')

# Zip the parquet data folder
shutil.make_archive('final_predictions_data', 'zip', 'final_predictions.parquet')

print("✅ Zip files created! Look in the folder icon on the left now.")

✅ Zip files created! Look in the folder icon on the left now.
